ЛР 2

In [220]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder, KBinsDiscretizer
from sklearn.model_selection import GridSearchCV

from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from catboost import CatBoostRegressor
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from pathlib import Path
from sklearn.feature_selection import SequentialFeatureSelector


from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error


In [159]:
df = pd.read_pickle("/home/mainuser/my_proj/data/clean_data.pkl")

df=df.rename(columns={'Selling_Price': 'target'})


In [160]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['target']), df['target'], test_size=0.25, random_state=42)

In [161]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   Car_Name       299 non-null    category
 1   Year           299 non-null    int64   
 2   target         299 non-null    float64 
 3   Present_Price  299 non-null    float64 
 4   Driven_kms     299 non-null    int64   
 5   Fuel_Type      299 non-null    category
 6   Selling_type   299 non-null    category
 7   Transmission   299 non-null    category
 8   Owner          299 non-null    int64   
 9   Car_Age        299 non-null    int64   
dtypes: category(4), float64(2), int64(4)
memory usage: 16.1 KB


In [162]:
X_train.iloc[0]

Car_Name            i10
Year               2011
Present_Price      4.43
Driven_kms        57000
Fuel_Type        Petrol
Selling_type     Dealer
Transmission     Manual
Owner                 0
Car_Age              15
Name: 225, dtype: object

In [163]:
cat_features = X_train.select_dtypes(include=['category','object']).columns.tolist()
cat_features

['Car_Name', 'Fuel_Type', 'Selling_type', 'Transmission']

In [164]:
num_features = X_train.select_dtypes(include=['int64','float64']).columns.tolist()
num_features

['Year', 'Present_Price', 'Driven_kms', 'Owner', 'Car_Age']

In [165]:
s_scaler = StandardScaler()
l_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
regressor = RandomForestRegressor(n_estimators=100, random_state=42)

In [166]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', s_scaler, num_features),
        ('cat', l_encoder, cat_features),
    ],
    remainder='drop'
)

In [167]:
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('model', regressor)])
pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [168]:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Year', 'Present_Price',
                                                   'Driven_kms', 'Owner', 'Car_Age']),
                                                   ('cat', OrdinalEncoder(handle_unknown='use_encoded_ value', unknown_value=-1),
                                                    ['Car_Name', 'Fuel_Type', 'Selling_Type', 'Transmission'])])),
                ('model', RandomForestRegressor(n_estimators=100, random_state=42))])


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [169]:
model_params = {
    'Car_Name': 'i10',
    'Year': 2011,
    'Present_Price': 4.43,
    'Driven_kms': 57000,
    'Fuel_Type': 'Petrol',
    'Selling_type': 'Dealer',
    'Transmission': 'Manual',
    'Owner': 0,
    'Car_Age': 15
}

df_pred = pd.DataFrame(model_params, index=[0])

In [170]:
pipeline.predict(df_pred)

array([2.653])

In [171]:
prediction = pipeline.predict(X_test)

metrics = {}
metrics["mae"] = mean_absolute_error(y_test, prediction)
metrics["mape"] = mean_absolute_percentage_error(y_test, prediction)
metrics["mse"] = mean_squared_error(y_test, prediction)

metrics

{'mae': 1.179589333333334,
 'mape': 0.3266406617213269,
 'mse': 9.442873176533338}

In [172]:
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

registry_uri = f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
tracking_uri = f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(registry_uri)

In [173]:
EXPERIMENT_NAME = "car_price_prediction_1"
RUN_NAME = "baseline_random_forest"
REGISTRY_MODEL_NAME = "car_price_rf"


In [174]:
input_example = X_train.head(5)
signature = infer_signature(model_input = X_train.head(5))

/home/mainuser/my_proj/.venv_my_proj/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


In [175]:
req_file = "/home/mainuser/my_proj/requirements.txt"

In [176]:
params_dict = pipeline.get_params()

In [177]:
experiment_id= mlflow.create_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.sklearn.log_model(pipeline,
                             artifact_path="models",
                             signature=signature,
                             input_example=input_example,
                             pip_requirements=req_file
                             )
    mlflow.log_metrics(metrics)
    mlflow.log_params(params_dict)

run = mlflow.get_run(run_id)
assert (run.info.status == "FINISHED")



2026/11/03 01:55:07 INFO mlflow.tracking._tracking_service.client: 🏃 View run baseline_random_forest at: http://127.0.0.1:5000/#/experiments/3/runs/6f4677125be44aacbb3c005e1e977153.
2026/11/03 01:55:07 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3.


In [179]:
num_features = X_train.select_dtypes(include=['number']).columns.tolist()


['Year', 'Present_Price', 'Driven_kms', 'Owner', 'Car_Age']

In [180]:
cat_features = X_train.select_dtypes(include=['category','object']).columns.tolist()


['Car_Name', 'Fuel_Type', 'Selling_type', 'Transmission']

In [190]:
bin_features = ['Driven_kms', 'Car_Age']

preprocessor_fe = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_features),
        ('bin', KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile'), bin_features)
    ],
    remainder='drop'
)

In [191]:
pipeline_fe = Pipeline(steps=[('preprocessor', preprocessor_fe),
                                ('model', RandomForestRegressor(
                                    n_estimators=100,
                                    random_state=42
                                ))
                            ])

pipeline_fe.fit(X_train, y_train)

/home/mainuser/my_proj/.venv_my_proj/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [192]:
X_train_fe_sklearn = pipeline_fe.named_steps[
    'preprocessor'
].transform(X_train)

feature_names = (
    pipeline_fe.named_steps['preprocessor']
    .get_feature_names_out()
    .tolist()
)

feature_names = [
    name.replace('bins_Driven_kms', 'Driven_kms_bin').replace('bins_Car_Age', 'Car_Age_bin') for name in feature_names
]

X_train_fe_sklearn = pd.DataFrame(X_train_fe_sklearn, columns=feature_names, index=X_train.index)

display(X_train_fe_sklearn.head())


,num__Year,num__Present_Price,num__Driven_kms,num__Owner,num__Car_Age,cat__Car_Name,cat__Fuel_Type,cat__Selling_type,cat__Transmission,bin__Driven_kms,bin__Car_Age
225,-0.938340,-0.340318,0.779750,-0.179605,0.938340,68.0,2.0,0.0,1.0,2.0,2.0
78,-0.592066,2.571009,1.892182,-0.179605,0.592066,66.0,1.0,0.0,0.0,2.0,2.0
66,-1.284614,1.441174,0.849277,-0.179605,1.284614,66.0,1.0,0.0,1.0,2.0,2.0
296,-1.630887,0.390294,1.855124,-0.179605,1.630887,55.0,2.0,0.0,1.0,2.0,2.0
192,-1.977161,-0.745435,0.536406,-0.179605,1.977161,15.0,2.0,1.0,1.0,2.0,2.0


In [196]:
feature_file = Path("/home/mainuser/my_proj/research/feature_names.txt")

feature_file.write_text(
    '\n'.join(feature_names),
    encoding='utf-8'
)



162

In [197]:
prediction_fe = pipeline_fe.predict(X_test)

metrics_fe = {
    'mae': mean_absolute_error(y_test, prediction_fe),
    'mape': mean_absolute_percentage_error(y_test, prediction_fe),
    'mse': mean_squared_error(y_test, prediction_fe)
}

comparison = pd.DataFrame([metrics, metrics_fe], index=['baseline', 'feature_engineered'])

display(comparison)

,mae,mape,mse
baseline,1.179589,0.326641,9.442873
feature_engineered,1.112144,0.314804,8.054458


In [198]:
input_example_fe = X_train.head(5)
signature_fe = infer_signature(model_input=input_example_fe)


/home/mainuser/my_proj/.venv_my_proj/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


In [200]:
mlflow.set_experiment(EXPERIMENT_NAME)
RUN_NAME_2="feature_engineered_random_forest"
params_dict_fe = pipeline_fe.named_steps['model'].get_params()
feature_file = Path("/home/mainuser/my_proj/research/feature_names.txt")
assert feature_file.is_file(), "Файл не найден"

with mlflow.start_run(run_name=RUN_NAME_2) as run:
    mlflow.log_params(params_dict_fe)
    mlflow.log_metrics(metrics_fe)
    mlflow.log_artifact(str(feature_file))
    mlflow.sklearn.log_model(sk_model=pipeline_fe,
                             artifact_path="model",
                             signature=signature_fe,
                             input_example=input_example_fe
                             )

    

2026/11/03 02:26:41 INFO mlflow.tracking._tracking_service.client: 🏃 View run feature_engineered_random_forest at: http://127.0.0.1:5000/#/experiments/3/runs/bcb2370269c2435294fad64b316ac004.
2026/11/03 02:26:41 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3.


In [202]:
preprocessor_fitted = pipeline_fe.named_steps['preprocessor']
feature_names_all = preprocessor_fitted.get_feature_names_out()
n_total=len(feature_names_all)
n_select = max(1, round(n_total * 0.55))

print("Всего признаков:", n_total)
print("Выбрано признаков:", n_select)

for i, name in enumerate(feature_names_all):
    print(i, name)

Всего признаков: 11
Выбрано признаков: 6
0 num__Year
1 num__Present_Price
2 num__Driven_kms
3 num__Owner
4 num__Car_Age
5 cat__Car_Name
6 cat__Fuel_Type
7 cat__Selling_type
8 cat__Transmission
9 bin__Driven_kms
10 bin__Car_Age


In [204]:
selector = SequentialFeatureSelector(
    estimator=RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    n_features_to_select=n_select,
    direction='forward',
    scoring='neg_mean_absolute_error',
    cv=3,
    n_jobs=-1
)

In [211]:
pipeline_3 = Pipeline(steps=[
    ('preprocessor', clone(pipeline_fe.named_steps['preprocessor'])),
    ('feature_selection', selector),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)) 
])

pipeline_3.fit(X_train, y_train)

/home/mainuser/my_proj/.venv_my_proj/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


,steps,"[('preprocessor', ...), ('feature_selection', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [213]:
feature_names = (pipeline_3.named_steps['preprocessor'].get_feature_names_out())

selected_mask = (pipeline_3.named_steps['feature_selection'].get_support())

selected_indices = np.flatnonzero(selected_mask)

selected_features = feature_names[selected_mask]

print("Выбранные признаки:")
for index, name in zip(selected_indices, selected_features):
    print(f"{index}: {name}")

print("Количество выбранных признаков:", len(selected_features))

Выбранные признаки:
0: num__Year
1: num__Present_Price
3: num__Owner
5: cat__Car_Name
8: cat__Transmission
10: bin__Car_Age
Количество выбранных признаков: 6


In [215]:
research_dir = Path("/home/mainuser/my_proj/research")
names_file = research_dir / "selected_feature_names.txt"

names_file.write_text(
    '\n'.join(selected_features.tolist()),
    encoding='utf-8'
)

indices_file = research_dir / "selected_indices.txt"
indices_file.write_text(
    '\n'.join(map(str, selected_indices)),
    encoding='utf-8'
)



12

In [ ]:
prediction_3 = pipeline_3.predict(X_test)

metrics_3 = {
    'mae': mean_absolute_error(y_test, prediction_3),
    'mape': mean_absolute_percentage_error(y_test, prediction_3),
    'mse': mean_squared_error(y_test, prediction_3)
}   

comparison_2 = pd.DataFrame([metrics, metrics_fe, metrics_3], index=['baseline', 'feature_engineered', 'feature_selection'])
display(comparison_2)



,mae,mape,mse
baseline,1.179589,0.326641,9.442873
feature_engineered,1.112144,0.314804,8.054458
feature_selection,1.052230,0.303070,6.176911


In [218]:
input_example_3 = X_train.head(5).copy()

signature_3 = infer_signature(model_input=input_example_3)

mlflow.set_experiment(EXPERIMENT_NAME)
RUN_NAME_3="feature_selection_random_forest"

with mlflow.start_run(run_name=RUN_NAME_3) as run:
    mlflow.log_params(pipeline_3.named_steps['model'].get_params())
    mlflow.log_params({
        "selection_method": "SFS_forward",
        "selected_feature_count": n_select,
        "total_feature_count": n_total
    })


    mlflow.log_metrics(metrics_3)
    mlflow.log_artifact(str(names_file))
    mlflow.log_artifact(str(indices_file))

    mlflow.sklearn.log_model(
        sk_model=pipeline_3,
        artifact_path="model",
        signature=signature_3,
        input_example=input_example_3
    )


/home/mainuser/my_proj/.venv_my_proj/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/03 02:53:17 INFO mlflow.tracking._tracking_service.client: 🏃 View run feature_selection_random_forest at: http://127.0.0.1:5000/#/experiments/3/runs/87b59ccf4e1543649b222948b792c4af.
2026/11/03 02:53:17 INFO mlflow

In [221]:
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [10, None],
    'model__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    estimator=clone(pipeline_3),
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=3,
    n_jobs=-1,
    verbose=2,
    refit = True
)
grid_search.fit(X_train, y_train)


Fitting 3 folds for each of 8 candidates, totalling 24 fits


/home/mainuser/my_proj/.venv_my_proj/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/home/mainuser/my_proj/.venv_my_proj/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/home/mainuser/my_proj/.venv_my_proj/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarnin

[CV] END model__max_depth=10, model__min_samples_split=2, model__n_estimators=200; total time=  16.3s


/home/mainuser/my_proj/.venv_my_proj/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


[CV] END model__max_depth=10, model__min_samples_split=2, model__n_estimators=100; total time=  16.8s
[CV] END model__max_depth=10, model__min_samples_split=2, model__n_estimators=100; total time=  16.9s


/home/mainuser/my_proj/.venv_my_proj/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/home/mainuser/my_proj/.venv_my_proj/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


... (15 outputs left out)

/home/mainuser/my_proj/.venv_my_proj/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/home/mainuser/my_proj/.venv_my_proj/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


[CV] END model__max_depth=None, model__min_samples_split=5, model__n_estimators=100; total time=  17.0s
[CV] END model__max_depth=None, model__min_samples_split=5, model__n_estimators=100; total time=  17.6s
[CV] END model__max_depth=None, model__min_samples_split=5, model__n_estimators=200; total time=  27.9s
[CV] END model__max_depth=None, model__min_samples_split=5, model__n_estimators=200; total time=  28.8s
[CV] END model__max_depth=None, model__min_samples_split=5, model__n_estimators=200; total time=  29.0s


/home/mainuser/my_proj/.venv_my_proj/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__max_depth': [10, None], 'model__min_samples_split': [2, 5], 'model__n_estimators': [100, 200]}"
,scoring,'neg_mean_absolute_error'
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...), ...]"


In [223]:
print("Лучшие параметры:")
print(grid_search.best_params_)
print("\nЛучшее MAE на кросс-валидации:")
print(-grid_search.best_score_)
pipeline_4 = grid_search.best_estimator_

print("\nИтоговая модель:")
print(pipeline_4)

Лучшие параметры:
{'model__max_depth': 10, 'model__min_samples_split': 2, 'model__n_estimators': 100}

Лучшее MAE на кросс-валидации:
0.7245254937836271

Итоговая модель:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Year', 'Present_Price',
                                                   'Driven_kms', 'Owner',
                                                   'Car_Age']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['Car_Name', 'Fuel_Type',
                                                   'Selling_type',
                                                   'Transmission']),
                                                 ('bi

In [224]:
prediction_4 = pipeline_4.predict(X_test)

metrics_4 = {}

metrics_4["mae"] = mean_absolute_error(y_test, prediction_4)
metrics_4["mape"] = mean_absolute_percentage_error(y_test, prediction_4)
metrics_4["mse"] = mean_squared_error(y_test, prediction_4)

metrics_4

{'mae': 1.1432528088139933,
 'mape': 0.32553647185894974,
 'mse': 7.918898970637701}

In [ ]:
comparison_3 = pd.DataFrame([ metrics_3, metrics_4], index=['feature_selection', 'tuned_model'])
display(comparison_3)


,mae,mape,mse
feature_selection,1.052230,0.303070,6.176911
tuned_model,1.143253,0.325536,7.918899


In [229]:
research_dir = Path("/home/mainuser/my_proj/research")
grid_results = pd.DataFrame(grid_search.cv_results_)
grid_file = research_dir / "grid_search_results.csv"
grid_results.to_csv(grid_file, index=False)
print("Файл сохранён:", grid_file)

Файл сохранён: /home/mainuser/my_proj/research/grid_search_results.csv


In [235]:
input_example_4 = X_train.head(5).copy()

signature_4 = infer_signature(model_input=input_example_4)

mlflow.set_experiment(EXPERIMENT_NAME)
RUN_NAME_4="random_forest_gridsearch"

mlflow.set_experiment(EXPERIMENT_NAME)
with mlflow.start_run(run_name=RUN_NAME_4) as run:
    mlflow.log_param("base_model", "random_forest_feature_selection")
    mlflow.log_params(grid_search.best_params_)
    mlflow.log_metrics(metrics_4)
    mlflow.log_metric(
        "cv_mae",
        -grid_search.best_score_
    )
    mlflow.log_artifact(str(grid_file))
    names_file = research_dir / "selected_feature_names.txt"
    indices_file = research_dir / "selected_indices.txt"

    mlflow.log_artifact(str(names_file))
    mlflow.log_artifact(str(indices_file))

    mlflow.sklearn.log_model(
        sk_model=pipeline_4,
        artifact_path="model",
        signature=signature_4,
        input_example=input_example_4,
        registered_model_name=REGISTRY_MODEL_NAME
    )


/home/mainuser/my_proj/.venv_my_proj/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Registered model 'car_price_rf' already exists. Creating a new version of this model...
2026/11/03 03:26:18 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. M

In [236]:
X_full = pd.concat(
    [X_train, X_test],
    axis=0
)

y_full = pd.concat(
    [y_train, y_test],
    axis=0
)

print("Размер X_train:", X_train.shape)
print("Размер X_test:", X_test.shape)
print("Размер полной выборки:", X_full.shape)
print("Количество целевых значений:", len(y_full))

Размер X_train: (224, 9)
Размер X_test: (75, 9)
Размер полной выборки: (299, 9)
Количество целевых значений: 299


In [237]:
pipeline_final = clone(pipeline_3)
pipeline_final.fit(X_full, y_full)

/home/mainuser/my_proj/.venv_my_proj/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


,steps,"[('preprocessor', ...), ('feature_selection', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [238]:
example_car = X_full.iloc[[0]]

prediction_final = pipeline_final.predict(example_car)
print("Предсказанная стоимоть:", prediction_final[0])

Предсказанная стоимоть: 2.5410000000000035


In [239]:
research_dir = Path("/home/mainuser/my_proj/research")
research_dir.mkdir(parents=True, exist_ok=True)

all_features_names = (
    pipeline_final.named_steps["preprocessor"]
    .get_feature_names_out()
)

selected_mask_final = (
    pipeline_final.named_steps["feature_selection"]
    .get_support()
)

selected_features_final = all_features_names[selected_mask_final]
selected_indices_final =np.flatnonzero(selected_mask_final)

print("Признаки финальной модели")

for index, name in zip(
    selected_indices_final,
    selected_features_final
):
    print(index, name)

names_final_file = research_dir / "final_selected_features.txt"

names_final_file.write_text(
    "\n".join(selected_features_final.tolist()),
    encoding="utf-8"
)



indices_final_file = research_dir / "final_selected_indices.txt"

indices_final_file.write_text(
    "\n".join(map(str, selected_indices_final)),
    encoding="utf-8"
)

Признаки финальной модели
0 num__Year
1 num__Present_Price
3 num__Owner
5 cat__Car_Name
8 cat__Transmission
10 bin__Car_Age


12

In [ ]:
input_example_final = X_full.head(5).copy()

signature_final = infer_signature(model_input=input_example_final)

mlflow.set_experiment(EXPERIMENT_NAME)
RUN_NAME_FINAL = 'final_feature_selection_random_forest'

params_final = (
    pipeline_final.named_steps["model"]
    .get_params()
)



/home/mainuser/my_proj/.venv_my_proj/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


In [244]:
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME_FINAL) as run:

    mlflow.log_params(params_final)
    mlflow.log_param(
        "training_dataset",
        "full_dataset"
    )
    mlflow.log_param(
        "training_samples",
        len(X_full)
    )
    mlflow.log_param(
        "feature_selection",
        "SequentialFeatureSelector"
    )

    mlflow.log_param(
        "selected_feature_count",
        len(selected_features_final)
    )

    mlflow.log_metrics({
        "previous_test_mae": metrics_3["mae"],
        "previous_test_mape": metrics_3["mape"],
        "previous_test_mse": metrics_3["mse"]
    })

    mlflow.log_artifact(str(names_final_file))
    mlflow.log_artifact(str(indices_final_file))

    mlflow.sklearn.log_model(
        sk_model=pipeline_final,
        artifact_path="model",
        signature=signature_final,
        input_example=input_example_final
    )

    final_run_id = run.info.run_id



2026/11/03 03:57:00 INFO mlflow.tracking._tracking_service.client: 🏃 View run final_feature_selection_random_forest at: http://127.0.0.1:5000/#/experiments/3/runs/754eecdc41994bbb9e5e408029a5e434.
2026/11/03 03:57:00 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3.


In [245]:
model_uri = f"runs:/{final_run_id}/model"

registered_model = mlflow.register_model(
    model_uri = model_uri,
    name = REGISTRY_MODEL_NAME
)

final_version = registered_model.version



Registered model 'car_price_rf' already exists. Creating a new version of this model...
2026/11/03 03:59:28 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: car_price_rf, version 3
Created version '3' of model 'car_price_rf'.
